# 10 - Class-level hazard ratios stratified by age, eGFR and BMI

Serious and minor bleeding hazard ratios for the drug classes within strata of age, eGFR and BMI (stratifier taken from observed values).

**Manuscript items:** Table S14

**Inputs:** derived cohort files from 01, comorbidity table from 02, CPRD DrugIssue extract, product dictionary, dosage lookup, interacting-drug code lists

**Outputs:** stratified tables displayed in the notebook

*Derived from the original analysis notebook `analysis_by_age_sex_bmi_ckd.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

In [ ]:
# Cap BLAS/OpenMP threads BEFORE numpy is imported. Without this each worker
# process below spawns its own thread pool and the machine thrashes.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import time
import matplotlib.pyplot as plt
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
from multiprocessing import Pool
from lifelines import CoxTimeVaryingFitter


In [ ]:
major_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_major.xlsx"), dtype=str)
major_doac_exposure["age_at_study_start"] = major_doac_exposure["age_at_study_start"].astype(int)
major_doac_exposure["study_start_date"] = pd.to_datetime(major_doac_exposure["study_start_date"])
major_doac_exposure["study_end_date"] = pd.to_datetime(major_doac_exposure["study_end_date"])
major_doac_exposure["event_date"] = pd.to_datetime(major_doac_exposure["event_date"])


#doac_exposure["event_date"] = pd.to_datetime(doac_exposure["event_date"])

major_doac_exposure = major_doac_exposure[(major_doac_exposure["age_at_study_start"] >= 18) &
                (major_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (major_doac_exposure["taken_warfarin_1w_before"] =="0") &
                 (major_doac_exposure["taken_warfarin_during_study"] =="0") ]

# age_at_study_start is retained here: it is the authoritative source for the
# backfill in the covariate cell below. Dropping it forced age to be re-derived
# from the comorbidity workbook, which is where the NaNs came from.
major_doac_exposure = major_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date", "age_at_study_start"]]
print(major_doac_exposure.shape[0])

In [ ]:
prod_dict = pd.read_csv(PRODUCT_DICT, sep="\t", dtype=str)
#prod_dict.dropna(subset=["DrugSubstanceName"], inplace=True)

In [ ]:
pats_data = pd.read_csv(PATIENT_FILE, sep=",", dtype=str)
pats_data["doac_start_date"] = pats_data["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
pats_data["doac_start_date"] = pd.to_datetime(pats_data["doac_start_date"], errors="coerce")
pats_data = pats_data[pats_data["patid"].isin(major_doac_exposure["patid"])]
pats_data["age"] = (pats_data["doac_start_date"] - pd.to_datetime(pats_data["yob"], format="%Y")).dt.days // 365.25
## add the comorbidities to major_doac_exposure
#major_doac_exposure["age"] = major_doac_exposure["patid"].map(pats_data.set_index("patid")["age"])

In [ ]:
a = pd.read_excel(os.path.join(CODE_LIST_DIR, "DOAC_DrugInteractions_Table.xlsx"), dtype=str)
a = a[["Drug", "Category", "Action", "PK/PD"]]

drugs_action_increase = a[a["Action"] == "Increased"]
drugs_action_decrease = a[a["Action"] == "Decreased"]
p_k = drugs_action_increase[drugs_action_increase["PK/PD"] == "PK"]
p_d = drugs_action_increase[drugs_action_increase["PK/PD"] == "PD"]

In [ ]:
antibiotic = drugs_action_increase[(drugs_action_increase["Category"]=="Antibiotic")]
ppi = a[(a["Category"]=="PPI")]
nsaid = drugs_action_increase[(drugs_action_increase["Category"]=="NSAID") ]
antifungal = drugs_action_increase[(drugs_action_increase["Category"]=="Antifungal") ] 
antiplatelet = drugs_action_increase[(drugs_action_increase["Category"]=="Antiplatelet" ) ]
ssri = drugs_action_increase[(drugs_action_increase["Category"]=="SSRI" ) & (drugs_action_increase["Action"]=="Increased")]
ccb = drugs_action_increase[(drugs_action_increase["Category"]=="CCB")]
snri = drugs_action_increase[(drugs_action_increase["Category"]=="SNRI")]
snri

In [ ]:
interacting_drugs = pd.read_excel(os.path.join(CODE_LIST_DIR, "DOAC_interacting_drugs.xlsx"), dtype=str)
interacting_drugs = interacting_drugs[interacting_drugs["DrugSubstanceName"].str.contains("/", case=False, na=False) == False]
interacting_drugs["DrugSubstanceName"] = np.where(interacting_drugs["DrugSubstanceName"].isna(), interacting_drugs["matched_drugs"], interacting_drugs["DrugSubstanceName"])

interacting_drugs = interacting_drugs[
    interacting_drugs.apply(
        lambda row: row["DrugSubstanceName"].startswith(row["matched_drugs"][:3])
        if pd.notna(row["DrugSubstanceName"]) else False,
        axis=1
    )
]
words_to_remove = ["eye drops", "inhaler", "cream", "ointment", "gel", "spray", "lotion", "shampoo", "cream"]
interacting_drugs = interacting_drugs[interacting_drugs["Term from EMIS"].str.contains("|".join(words_to_remove), case=False, na=False) == False]
interacting_drugs_codes = interacting_drugs[["ProdCodeId", "matched_drugs"]]
interacting_drugs_codes.rename(columns={"matched_drugs": "drug_name", "ProdCodeId":"prodcodeid"}, inplace=True)
interacting_drugs_codes.drop_duplicates(subset=["prodcodeid"], inplace=True)
interacting_drugs_codes["drug_class"] = interacting_drugs_codes["drug_name"].map(a.set_index("Drug")["Category"])
interacting_drugs_codes

In [ ]:
def calculate_enddate(df, dosage):
    df = df.copy()
    df["daily_dose"] = df["dosageid"].map(dosage.set_index("dosageid")["daily_dose"])
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["daily_dose"] = pd.to_numeric(df["daily_dose"], errors="coerce").fillna(0)

    # duration==0 rows that CAN be resolved from quantity / daily dose
    mask1 = (
        (df["duration"] == 0)
        & df["quantity"].notna()
        & (df["daily_dose"] != 0)
        & (df["quantity"] != 0)
    )
    # All other duration==0 rows (daily_dose unknown/zero, OR quantity missing/zero)
    # are imputed from the per-product median below, rather than being silently
    # left at 0 (which would otherwise produce a same-day enddate for an
    # unknown/unresolvable duration).
    mask2 = (df["duration"] == 0) & (~mask1)

    df["new_duration"] = np.where(mask1, df["quantity"] / df["daily_dose"], df["duration"])
    df["new_duration"] = pd.to_numeric(df["new_duration"], errors="coerce")

    # Median computed only from rows with a genuinely recorded (non-zero) duration,
    # so neither mask1-resolved rows nor other zero-duration rows distort it.
    median_durations = (
        df.loc[df["duration"] != 0].groupby("prodcodeid")["duration"].median()
    )

    df["new_duration"] = np.where(mask2, df["prodcodeid"].map(median_durations), df["new_duration"])

    df["enddate"] = (pd.to_datetime(df["issuedate"]) + pd.to_timedelta(df["new_duration"], unit="D"))

    return df

In [ ]:
import zipfile
dosage = pd.read_csv(os.path.join(LOOKUP_DIR, "common_dosages.txt"), sep="\t",dtype=str)


zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Filtering
    df = df[(df['prodcodeid'].isin(interacting_drugs_codes["prodcodeid"])) & (df['patid'].isin(major_doac_exposure["patid"].tolist()))]

    df["study_start_date"] = df["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
    df["study_end_date"] = df["patid"].map(major_doac_exposure.set_index("patid")["study_end_date"])
    
    df = calculate_enddate(df, dosage)

    df = df[(df["issuedate"] < df["study_end_date"]) &(df["enddate"] > df["study_start_date"])]

    return df


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    # Run in parallel
    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    # Combine results
    interacting_drugs_exposure = pd.concat(all_results, ignore_index=True)

interacting_drugs_exposure["enddate"] = pd.to_datetime(interacting_drugs_exposure["enddate"], errors='coerce')
interacting_drugs_exposure["enddate"] = interacting_drugs_exposure["enddate"] + pd.to_timedelta(1, unit='D')
interacting_drugs_exposure.drop(columns=["dosageid", "duration", "quantity", "daily_dose", "probobsid", "new_duration"], inplace=True)
interacting_drugs_exposure.sort_values(['patid', 'issuedate'], inplace=True)
interacting_drugs_exposure.reset_index(drop=True, inplace=True)
interacting_drugs_exposure["drug_class"] = interacting_drugs_exposure["prodcodeid"].map(interacting_drugs_codes.set_index("prodcodeid")["drug_class"])
interacting_drugs_exposure["drug_name"] = interacting_drugs_exposure["prodcodeid"].map(interacting_drugs_codes.set_index("prodcodeid")["drug_name"])
interacting_drugs_exposure = interacting_drugs_exposure[interacting_drugs_exposure["drug_name"].isin(drugs_action_increase["Drug"].tolist())]

In [ ]:
major_interacting_drugs_presc = interacting_drugs_exposure.copy()
major_interacting_drugs_presc["study_end_date"] = major_interacting_drugs_presc["patid"].map(major_doac_exposure.set_index("patid")["study_end_date"])
major_interacting_drugs_presc = major_interacting_drugs_presc[(major_interacting_drugs_presc["issuedate"] < major_interacting_drugs_presc["study_end_date"]) ]
major_interacting_drugs_presc["enddate"] = np.where(major_interacting_drugs_presc["enddate"] > major_interacting_drugs_presc["study_end_date"], 
                                                    major_interacting_drugs_presc["study_end_date"], major_interacting_drugs_presc["enddate"])
major_interacting_drugs_presc.reset_index(drop=True, inplace=True)
major_interacting_drugs_presc["drug_term"] = major_interacting_drugs_presc["prodcodeid"].map(prod_dict.set_index("ProdCodeId")["Term from EMIS"])
major_interacting_drugs_presc

In [ ]:
major_events_df = major_doac_exposure[major_doac_exposure["event_date"].notnull()]

In [ ]:
# Keep only required columns
presc = major_interacting_drugs_presc[["patid", "issuedate", "enddate", "drug_class"]]
events = major_events_df.reset_index(names="event_row")

merged = events.merge(presc,on="patid",how="left")
active = merged[(merged["issuedate"] <= merged["event_date"]) &(merged["enddate"] >= merged["event_date"])]
active_drugs = (active.groupby("event_row")["drug_class"].agg(list))
major_events_df["active_drugs_ls"] = (major_events_df.index.map(active_drugs))
major_events_df["active_drugs_ls"] = (major_events_df["active_drugs_ls"].apply(lambda x: x if isinstance(x, list) else []))

major_events_df

In [ ]:
from collections import Counter

drug_presc_counts = major_interacting_drugs_presc.groupby("drug_class")["patid"].nunique().reset_index()
drug_presc_counts.rename(columns={"patid": "num_patients"}, inplace=True)
drug_presc_counts.sort_values (by="num_patients", ascending=False, inplace=True)


drug_counts = Counter()
for drugs in major_events_df["active_drugs_ls"]:
    drug_counts.update(drugs)

drug_counts_df = pd.DataFrame.from_dict(drug_counts, orient='index', columns=['count']).reset_index().rename(columns={'index': 'drug_class'})
drug_counts_df = drug_counts_df.sort_values("count", ascending=False).reset_index(drop=True)


drug_counts_df["presc_count"] = drug_counts_df["drug_class"].map(drug_presc_counts.set_index("drug_class")["num_patients"])
drug_counts_df

In [ ]:
drug_classes_to_consider = ["NSAID", "Antiplatelet", "SSRI", "CCB", "SNRI", "Antiarrythmic"]

drug_counts_df = drug_counts_df[drug_counts_df["drug_class"].isin(drug_classes_to_consider)].reset_index(drop=True)
drug_counts_df

In [ ]:
interacting_drugs_exposure = interacting_drugs_exposure[interacting_drugs_exposure["drug_class"].isin(drug_classes_to_consider)].reset_index(drop=True)

In [ ]:
interacting_drugs_exposure

In [ ]:
def process_group(group):
    group = group.sort_values('issuedate').reset_index(drop=True)
    merged = []

    # Set days_gap based on drug_class
    drug_class = group['drug_class'].iloc[0]  # Assuming all rows in the group have the same drug_class
    if drug_class == 'Antibiotic':
        days_gap = 7
    elif drug_class == 'Antifungal':
        days_gap = 7
    else:
        days_gap = 14  # default

    current_start = group.loc[0, 'issuedate']
    current_end = group.loc[0, 'enddate']

    for i in range(1, len(group)):
        next_start = group.loc[i, 'issuedate']
        next_end = group.loc[i, 'enddate']

        if (next_start - current_end).days < days_gap:
            current_end = max(current_end, next_end)
        else:
            merged.append({'patid': group.loc[i-1, 'patid'],
                           'drug_class': group.loc[i-1, 'drug_class'],
                           'issuedate': max(current_start, group['study_start_date'].iloc[0]),
                           'enddate': min(current_end, group['study_end_date'].iloc[0])})
            current_start = next_start
            current_end = next_end

    merged.append({'patid': group.loc[len(group)-1, 'patid'],
                   'drug_class': group.loc[len(group)-1, 'drug_class'],
                   'issuedate': max(current_start, group['study_start_date'].iloc[0]),
                   'enddate': min(current_end, group['study_end_date'].iloc[0])})
    return pd.DataFrame(merged)

def merge_prescriptions(df):
    # Ensure dates are datetime
    for col in ['issuedate', 'enddate', 'study_start_date', 'study_end_date']:
        df[col] = pd.to_datetime(df[col])

    # Split groups
    groups = [group for _, group in df.groupby(['patid', 'drug_class'])]

    # Multiprocessing
    with Pool(50) as pool:
        result_list = pool.map(process_group, groups)

    merged_df = pd.concat(result_list, ignore_index=True)
    return merged_df

In [ ]:
major_interacting_drugs_presc_merged = merge_prescriptions(major_interacting_drugs_presc)
major_interacting_drugs_presc_merged["study_start_date"] = major_interacting_drugs_presc_merged["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
major_interacting_drugs_presc_merged["study_end_date"] = major_interacting_drugs_presc_merged["patid"].map(major_doac_exposure.set_index("patid")["study_end_date"])
#interacting_drugs_presc_merged["event_date"] = interacting_drugs_presc_merged["patid"].map(events_df.set_index("patid")["event_date"])

major_interacting_drugs_presc_merged

In [ ]:
def _process_patient(args):
    patid, group = args
    
    doac_start = group["study_start_date"].iloc[0]
    doac_end = group["study_end_date"].iloc[0]

    # clip exposures to DOAC period
    group = group.copy()
    group["start"] = group["issuedate"].clip(lower=doac_start, upper=doac_end)
    group["stop"] = group["enddate"].clip(lower=doac_start, upper=doac_end)

    # collect cut points
    cut_points = sorted(set([doac_start, doac_end]) | set(group["start"]) | set(group["stop"]))
    intervals = [(cut_points[i], cut_points[i+1]) for i in range(len(cut_points)-1)]

    rows = []
    for start, stop in intervals:
        if start >= stop:
            continue
        active = group[(group["start"] <= start) & (group["stop"] >= stop)]
        active_drugs = sorted(active["drug_class"].unique().tolist())
        rows.append({
            "patid": patid,
            "start_date": start,
            "end_date": stop,
            "concomitant_drugs_ls": active_drugs
        })
    return rows

def build_exposure_timeline(df, n_jobs=None):
    df = df.copy()
    df['issuedate'] = pd.to_datetime(df['issuedate'])
    df['enddate'] = pd.to_datetime(df['enddate'])
    df['study_start_date'] = pd.to_datetime(df['study_start_date'])
    df['study_end_date'] = pd.to_datetime(df['study_end_date'])
    
    grouped = list(df.groupby("patid"))

    with mp.Pool(processes=60) as pool:
        results = pool.map(_process_patient, grouped)

    # flatten list of lists
    flat_results = [row for sublist in results for row in sublist]
    return pd.DataFrame(flat_results)

In [ ]:
major_exposure_timeline_df = build_exposure_timeline(major_interacting_drugs_presc_merged)

major_exposure_timeline_df["study_start_date"] = major_exposure_timeline_df["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
major_exposure_timeline_df["study_end_date"] = major_exposure_timeline_df["patid"].map(major_doac_exposure.set_index("patid")["study_end_date"])

major_exposure_timeline_df["event_date"] = major_exposure_timeline_df["patid"].map(major_events_df.set_index("patid")["event_date"])
major_exposure_timeline_df["event_flag"] = np.where(
    (major_exposure_timeline_df["event_date"] >= major_exposure_timeline_df["start_date"]) & 
    (major_exposure_timeline_df["event_date"] <= major_exposure_timeline_df["end_date"]), 1, 0)

In [ ]:
pats_with_exposures = set(major_exposure_timeline_df["patid"].unique())

unique_major_events_df = major_events_df.drop_duplicates(subset=["patid"]).copy()

pats_without_exposures_df = major_doac_exposure[~major_doac_exposure["patid"].isin(pats_with_exposures)].copy()
pats_without_exposures_df["concomitant_drugs_ls"] = [[] for _ in range(len(pats_without_exposures_df))]
pats_without_exposures_df["start_date"] = pats_without_exposures_df["study_start_date"]
pats_without_exposures_df["end_date"] = pats_without_exposures_df["study_end_date"]
pats_without_exposures_df["event_date"] = pats_without_exposures_df["patid"].map(
    unique_major_events_df.set_index("patid")["event_date"]
)
pats_without_exposures_df["event_flag"] = np.where(
    (pats_without_exposures_df["event_date"] >= pats_without_exposures_df["start_date"]) &
    (pats_without_exposures_df["event_date"] <= pats_without_exposures_df["end_date"]),
    1, 0
)

major_exposure_timeline_df = pd.concat([major_exposure_timeline_df, pats_without_exposures_df], ignore_index=True)
major_exposure_timeline_df.reset_index(drop=True, inplace=True)

In [ ]:
major_df = major_exposure_timeline_df.copy()
major_df = major_df[major_df["start_date"] < major_df["end_date"]].reset_index(drop=True)

for drug in drug_classes_to_consider:
    major_df[f"{drug}"] = major_df["concomitant_drugs_ls"].apply(lambda drugs: 1 if drug in drugs else 0)

In [ ]:
prod_dict = pd.read_csv(PRODUCT_DICT, sep="\t", dtype=str)

ppi_drug_codes = prod_dict[prod_dict["drug_class"].str.contains("Proton Pump Inhibitor", case=False, na=False)]
ppi_drug_codes = ppi_drug_codes[~ppi_drug_codes["DrugSubstanceName"].str.contains("/", case=False, na=False)]["ProdCodeId"].unique().tolist()

zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Ensure datetime conversion early
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')

    # Filter rows first (KEEP dataframe)
    df = df[
        df['prodcodeid'].isin(ppi_drug_codes) &
        df['patid'].isin(major_doac_exposure["patid"])
    ].copy()

    # Map study start date
    df["study_start_date"] = df["patid"].map(
        major_doac_exposure.set_index("patid")["study_start_date"]
    )

    df["study_start_date"] = pd.to_datetime(df["study_start_date"], errors='coerce')

    # Apply time window filter
    df = df[
        (df["issuedate"] >= df["study_start_date"] - pd.Timedelta(days=60)) &
        (df["issuedate"] < df["study_start_date"])
    ]

    return df["patid"].unique().tolist()


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    ppi_pats = set(p for sublist in all_results for p in sublist)

In [ ]:
major_df["PPI_baseline"] = major_df["patid"].isin(ppi_pats).astype(int)

In [ ]:
pats_data = pd.read_csv(PATIENT_FILE, sep=",", dtype=str)
pats_data["gender"] = pats_data["gender"].map({"1": 1, "2": 0})

In [ ]:
# ===========================================================================
# Baseline covariates - RAW values, missing continuous values left as NaN
# ===========================================================================
# The comorbidity workbook is already the raw one here. What changes is that the
# missing continuous values are now deliberately preserved so the multiple
# imputation below can handle them, and age is backfilled rather than imputed.

comorbidities = pd.read_excel(os.path.join(DERIVED_DIR, "pats_comorbidities_DDI.xlsx"), dtype={"patid": str})

binary_comorbidity_cols = ["major_bled_hist", "hypertension", "liver_disease", "stroke", "gender", "heart_failure",
                           "atrial_fibrillation", "venous_thromboembolism", "cancer", "diabetes", "smoking", "alcohol"]
continuous_comorbidity_cols = ["age_at_study_start", "ckd_value", "bmi_value"]

comorbidities["gender"] = comorbidities["patid"].map(pats_data.set_index("patid")["gender"])

for col in continuous_comorbidity_cols:
    if col in comorbidities.columns:
        comorbidities[col] = pd.to_numeric(comorbidities[col], errors="coerce")


def backfill_age(df, cohort_df, label=""):
    """Age is derivable exactly from year of birth and the index date, so it is
    merged in rather than imputed. Imputing a variable you can compute would not
    survive review."""
    before = int(df["age_at_study_start"].isna().sum()) if "age_at_study_start" in df else len(df)
    age = pd.to_numeric(df.get("age_at_study_start"), errors="coerce")
    if "age_at_study_start" in cohort_df.columns:
        lut = cohort_df.drop_duplicates("patid").set_index("patid")["age_at_study_start"]
        age = age.fillna(df["patid"].map(pd.to_numeric(lut, errors="coerce")))
    if age.isna().any() and "age" in pats_data.columns:
        lut2 = pats_data.drop_duplicates("patid").set_index("patid")["age"]
        age = age.fillna(df["patid"].map(pd.to_numeric(lut2, errors="coerce")))
    df["age_at_study_start"] = age
    after = int(age.isna().sum())
    print(f"[{label}] age missing rows: {before:,} -> {after:,}"
          + ("  <-- still unresolved, investigate before imputing" if after else ""))
    return df


def report_missingness(df, label=""):
    """Patient-level missingness. Cross-check against the manuscript (eGFR 3.7%,
    BMI 8.4%): a mismatch points at the merge, not at the data."""
    pat = df.drop_duplicates("patid")
    rep = pd.DataFrame([
        {"variable": c, "n_missing": int(pat[c].isna().sum()),
         "pct_missing": round(100 * pat[c].isna().mean(), 2)}
        for c in continuous_comorbidity_cols])
    print(f"[{label}] {pat['patid'].nunique():,} patients")
    print(rep.to_string(index=False))
    return rep


_absent = set(major_df["patid"]) - set(comorbidities["patid"])
print(f"Patients missing from the comorbidity workbook entirely: {len(_absent):,} "
      f"({100 * len(_absent) / major_df['patid'].nunique():.2f}%)")

for col in binary_comorbidity_cols:
    major_df[col] = major_df["patid"].map(comorbidities.set_index("patid")[col]).fillna(0).astype(int)

# NaNs deliberately carried through - the imputation step fills them, and the
# strata below are built from the ORIGINAL observed values.
for col in continuous_comorbidity_cols:
    major_df[col] = major_df["patid"].map(comorbidities.set_index("patid")[col])

major_df = backfill_age(major_df, major_doac_exposure, "Serious bleeding")
missing_report_major = report_missingness(major_df, "Serious bleeding")


In [ ]:
# ===========================================================================
# PPI adjustment (NSAID only) + parallel stratum fitting
# ===========================================================================
# A single covariate in one joint model necessarily adjusts every class's hazard
# ratio, so every stratum model below is fitted twice - once with PPI_baseline
# and once without - and each reported row is taken from the appropriate fit.
# Only NSAID is adjusted; aspirin has no term of its own here, it sits inside
# the Antiplatelet class alongside clopidogrel, which is left unadjusted.
#
# Every stratum is an independent fit, so they all go into one pool per section.
# Workers filter the shared frame themselves rather than being handed a subset:
# the frame is published to a module-level global BEFORE the Pool is created, so
# on Linux the forked workers inherit it copy-on-write and nothing large is ever
# pickled. Requires the default 'fork' start method.
#
# NOTE: the original notebook reassigned `comorbidities_covariates` in each
# section (dropping age, then ckd, then bmi) and the fit loops read it at call
# time, so re-running an earlier section after a later one silently fitted the
# wrong covariate set. build_formulas() below takes the exclusion explicitly, so
# each section is self-contained and order no longer matters.

PPI_ADJUSTED_CLASSES = ["NSAID"]
FIT_N_JOBS = 12

_missing = [c for c in PPI_ADJUSTED_CLASSES if c not in drug_classes_to_consider]
if _missing:
    raise ValueError(f"PPI_ADJUSTED_CLASSES not in drug_classes_to_consider: {_missing}")
print("Classes adjusted for baseline PPI use:", PPI_ADJUSTED_CLASSES)


def ppi_flag(drug):
    return "Yes" if drug in PPI_ADJUSTED_CLASSES else "No"


def build_formulas(exclude_continuous=()):
    """Return (formula_without_ppi, formula_with_ppi, comorbidity_columns).

    `exclude_continuous` drops the continuous covariate that the section is
    stratifying on, since it would be collinear within a stratum.
    """
    comorbs = binary_comorbidity_cols + [
        c for c in continuous_comorbidity_cols if c not in exclude_continuous]
    base = " + ".join(drug_classes_to_consider + comorbs)
    return base, base + " + PPI_baseline", comorbs


_FIT_DF = None   # inherited by forked workers, never pickled


def _fit_worker(task):
    key, formula, cols, filt = task
    df = _FIT_DF
    if filt is not None:
        filter_col, filter_val = filt
        df = df[df[filter_col] == filter_val]
    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        df[cols],
        id_col='patid', start_col='start_date', stop_col='end_date',
        event_col='event_flag', formula=formula, show_progress=False,
    )
    return key, {"summary": ctv.summary, "n_rows": len(df)}


def fit_models(df, specs, label="", n_jobs=FIT_N_JOBS):
    """specs: {key: (formula, cols, filter_or_None)} -> {key: {summary, n_rows}}."""
    global _FIT_DF
    tasks = [(k, f, c, flt) for k, (f, c, flt) in specs.items()]
    t0 = time.time()
    _FIT_DF = df
    try:
        with Pool(min(n_jobs, len(tasks))) as pool:
            results = pool.map(_fit_worker, tasks)
    finally:
        _FIT_DF = None
    print(f"[{label}] fitted {len(results)} models in {time.time()-t0:.1f}s")
    return dict(results)


def split_summary(fits, group):
    """One stratum's table, each class taken from the model appropriate to it."""
    rows = []
    for drug in drug_classes_to_consider:
        suffix = "with_ppi" if drug in PPI_ADJUSTED_CLASSES else "no_ppi"
        summary = fits[f"{group}|{suffix}"]["summary"]
        if drug not in summary.index:
            continue
        r = summary.loc[drug]
        rows.append({
            "covariate": drug,
            "PPI adjusted": ppi_flag(drug),
            "HR": round(r["exp(coef)"], 4),
            "CI Lower": round(r["exp(coef) lower 95%"], 4),
            "CI Upper": round(r["exp(coef) upper 95%"], 4),
            "p": round(r["p"], 4),
        })
    return pd.DataFrame(rows).reset_index(drop=True)


def fit_strata(df, group_col, groups, exclude_continuous=(), label=""):
    """Fit every stratum twice (with/without PPI) in one pool.

    Returns {group: summary DataFrame} in the same shape the old serial loops
    produced, so the downstream table-building cells are unchanged.
    """
    f_no, f_yes, comorbs = build_formulas(exclude_continuous)
    cols = (["patid", "start_date", "end_date", "event_flag"]
            + drug_classes_to_consider + comorbs)

    specs = {}
    for g in groups:
        specs[f"{g}|no_ppi"] = (f_no, cols, (group_col, g))
        specs[f"{g}|with_ppi"] = (f_yes, cols + ["PPI_baseline"], (group_col, g))

    fits = fit_models(df, specs, label=label)
    return {g: split_summary(fits, g) for g in groups}


## Multiple imputation of missing continuous covariates

Missing values in the continuous baseline covariates (`ckd_value` / eGFR and
`bmi_value`) are handled by **multiple imputation by chained equations (MICE)**
using `miceforest` (random forest / LightGBM chained equations with predictive
mean matching), rather than by complete-case analysis or a missing-indicator
category.

Three things make this survival-analysis-appropriate rather than a generic MICE run:

1. **The outcome is in the imputation model.** The event indicator, follow-up
   time and the **Nelson-Aalen cumulative hazard** estimate are included as
   predictors, following White IR & Royston P, *Stat Med* 2009;28:1982-1998.
   Omitting the outcome biases covariate-outcome associations toward the null.
2. **The imputation model is congenial with the substantive model.** Every
   covariate in the Cox models is a predictor in the imputation model, plus
   ever-exposed indicators for each interacting drug and baseline PPI use, so
   the imputation model is at least as rich as the analysis model.
3. **Estimates are pooled by Rubin's rules** on the log-hazard scale, with
   Barnard-Rubin degrees of freedom, and the fraction of missing information
   (FMI) is reported for each coefficient.

Imputation is run **separately for each outcome cohort** (serious, minor, all
bleeding), because the outcome and follow-up time entering the imputation model
differ between them.

A **complete-case sensitivity analysis** is run alongside, as recommended by
Sterne JAC et al., *BMJ* 2009;338:b2393 and required for STROBE / RECORD-PE
reporting.


In [ ]:
# ===========================================================================
# Multiple imputation helpers (MICE + Rubin's rules)
# ===========================================================================
from scipy import stats
from lifelines import NelsonAalenFitter

# ---- Configuration --------------------------------------------------------
# m = number of imputed datasets. Set to 10 to match the manuscript and the
# White/Royston/Wood (2011) rule of thumb that m should be at least the
# percentage of incomplete cases (BMI ~8.4% missing). Set to 5 if you prefer
# the smaller number - nothing else needs to change.
MI_CONFIG = dict(
    m=10,
    iterations=5,
    seed=20240824,
    mean_match_candidates=5,   # predictive mean matching: imputed values are
                               # drawn from observed values of near neighbours
    n_jobs=10,                 # imputed datasets are independent draws, so each
                               # one is built in its own worker process. Set to 1
                               # to fall back to a single m-dataset kernel.
)
IMPUTE_COLS = ["ckd_value", "bmi_value"]   # continuous covariates to impute

# The 2 x m adjusted fits per outcome are all independent, so they are farmed
# out as a single flat pool of tasks rather than looping over imputations. Each
# worker builds its own copy of the design frame, so peak memory is roughly
# MI_FIT_N_JOBS x (size of the interval-level frame restricted to model
# columns). Lower this first if the node runs out of memory.
MI_FIT_N_JOBS = 20

# NOTE ON THREADS: the first cell of this notebook pins OMP_NUM_THREADS=1, which
# also silences LightGBM's internal threading inside miceforest. That is the
# right default here because MICE is parallelised across processes instead.
# If you set MI_CONFIG["n_jobs"]=1, give LightGBM threads back explicitly:
#     run_mice(..., num_threads=16)      # **lgb_kwargs are forwarded to mice()
# Do not do both at once - the two layers will oversubscribe the cores.


def check_mi_versions():
    """miceforest 6.0.x is incompatible with lightgbm >= 4.6 (the
    Booster.__inner_predict signature changed). Fail loudly and early rather
    than 40 minutes into a run."""
    import miceforest, lightgbm
    print(f"miceforest {miceforest.__version__} | lightgbm {lightgbm.__version__}")
    major_minor = tuple(int(x) for x in lightgbm.__version__.split(".")[:2])
    if major_minor >= (4, 6):
        print("WARNING: kernel.mice() may raise "
              "'Booster.__inner_predict() takes 1 positional argument'. "
              "If it does, run:  pip install 'lightgbm<4.6'")


# ---- 1. Nelson-Aalen cumulative hazard (White & Royston 2009) --------------
def nelson_aalen_cumhaz(durations, events):
    durations = np.asarray(durations, dtype=float)
    events = np.asarray(events, dtype=int)
    naf = NelsonAalenFitter()
    naf.fit(durations, event_observed=events)
    ch = naf.cumulative_hazard_.iloc[:, 0]
    times = ch.index.to_numpy(dtype=float)
    idx = np.clip(np.searchsorted(times, durations, side="right") - 1, 0, len(times) - 1)
    return ch.to_numpy()[idx]


# ---- 2. Patient-level frame for the imputation model ----------------------
def build_imputation_frame(analysis_df, comorbidities, binary_cols, continuous_cols,
                           drug_terms=(), extra_binary=("PPI_baseline",),
                           event_col="event_flag"):
    """One row per patient: total follow-up, event indicator, Nelson-Aalen
    cumulative hazard, every analysis covariate, and ever-exposed flags for the
    interacting drugs (so the imputation model is congenial with the Cox model)."""
    df = analysis_df
    terms = [t for t in list(drug_terms) + list(extra_binary) if t in df.columns]

    slim = pd.DataFrame({
        "patid": df["patid"].to_numpy(),
        "_dur": (df["end_date"] - df["start_date"]).dt.days.clip(lower=0).to_numpy(),
        "event_flag": df[event_col].to_numpy(),
    })
    for t in terms:
        slim[t] = df[t].to_numpy().astype("int8")

    agg_map = {"fu_days": ("_dur", "sum"), "event": ("event_flag", "max")}
    agg_map.update({f"ever_{t}": (t, "max") for t in terms})
    pat = slim.groupby("patid", sort=False).agg(**agg_map).reset_index()

    pat["fu_years"] = (pat.pop("fu_days") / 365.25).clip(lower=1 / 365.25)
    pat["event"] = pat["event"].astype(int)
    pat["na_cumhaz"] = nelson_aalen_cumhaz(pat["fu_years"], pat["event"])
    pat["log_fu_years"] = np.log(pat["fu_years"])
    for t in terms:
        pat[f"ever_{t}"] = pat[f"ever_{t}"].astype(float)

    # Covariate values are taken from the ANALYSIS frame, not from the workbook.
    # That frame is what the Cox model actually sees, and it carries any
    # correction applied after the merge - notably the age backfill. Reading from
    # the workbook here would silently undo those corrections and hand MICE a
    # column it was never meant to impute. The workbook is only a fallback for
    # columns the analysis frame does not carry.
    first = analysis_df.drop_duplicates("patid").set_index("patid")
    com = (comorbidities.drop_duplicates(subset=["patid"]).set_index("patid")
           if comorbidities is not None else None)

    def _pull(col):
        if col in first.columns:
            return pd.to_numeric(pat["patid"].map(first[col]), errors="coerce")
        if com is not None and col in com.columns:
            return pd.to_numeric(pat["patid"].map(com[col]), errors="coerce")
        return None

    for col in binary_cols:
        v = _pull(col)
        if v is not None:
            pat[col] = v.fillna(0).astype(float)
    for col in continuous_cols:
        v = _pull(col)
        if v is not None:
            pat[col] = v

    return pat


# ---- 3. MICE with miceforest ----------------------------------------------
def _make_kernel(work, m, seed, mean_match_candidates=5):
    """Construct an ImputationKernel across miceforest 5.x / 6.x signatures."""
    import miceforest as mf
    attempts = [
        dict(num_datasets=m, random_state=seed, mean_match_candidates=mean_match_candidates,
             save_all_iterations_data=True),
        dict(datasets=m, random_state=seed, mean_match_candidates=mean_match_candidates,
             save_all_iterations=True),
        dict(num_datasets=m, random_state=seed),
        dict(datasets=m, random_state=seed),
    ]
    last = None
    for kw in attempts:
        try:
            return mf.ImputationKernel(work, **kw)
        except TypeError as exc:
            last = exc
    raise RuntimeError(f"Could not construct miceforest ImputationKernel: {last}")


_MICE_WORK = None       # published to forked workers, never pickled
_MICE_TARGETS = None


def _mice_worker(args):
    """One imputed dataset, built by its own single-dataset kernel."""
    idx, seed, iterations, mmc, lgb_kwargs = args
    kernel = _make_kernel(_MICE_WORK, 1, seed, mmc)
    kernel.mice(iterations, **lgb_kwargs)
    comp = kernel.complete_data(dataset=0)
    return idx, {c: np.asarray(comp[c], dtype=float) for c in _MICE_TARGETS}


def run_mice_parallel(pat_frame, continuous_cols, targets, m, iterations, seed,
                      mean_match_candidates, n_jobs, label="", **lgb_kwargs):
    """m independent chains in m worker processes.

    Statistically this is the same estimator as one m-dataset kernel: the m
    imputations are independent draws either way, the only difference being that
    each chain gets its own RNG stream. Streams are spawned from a single
    SeedSequence so the whole run is reproducible from MI_CONFIG["seed"]."""
    global _MICE_WORK, _MICE_TARGETS
    t0 = time.time()
    ss = np.random.SeedSequence(seed)
    seeds = [int(s.generate_state(1)[0]) % (2 ** 31 - 1) for s in ss.spawn(m)]
    tasks = [(i, seeds[i], iterations, mean_match_candidates, lgb_kwargs) for i in range(m)]

    work = pat_frame.drop(columns=["patid"]).copy()
    work = work.apply(pd.to_numeric, errors="coerce").astype("float64")

    _MICE_WORK, _MICE_TARGETS = work, targets
    try:
        with Pool(min(n_jobs, m)) as pool:
            results = dict(pool.map(_mice_worker, tasks))
    finally:
        _MICE_WORK = _MICE_TARGETS = None

    out = []
    for i in range(m):
        frame = pd.DataFrame({"patid": pat_frame["patid"].to_numpy()})
        for c in continuous_cols:
            if c in results[i]:
                frame[c] = results[i][c]
            elif c in pat_frame.columns:
                frame[c] = pat_frame[c].to_numpy()
        out.append(frame)
    print(f"[{label}] MICE: {m} datasets on {min(n_jobs, m)} workers "
          f"in {time.time() - t0:.1f}s | seeds {seeds[:3]}...")
    return out, seeds


def run_mice(pat_frame, continuous_cols, m=10, iterations=5, seed=20240824,
             mean_match_candidates=5, n_jobs=1, label="", verbose=True, **lgb_kwargs):
    """Returns (list of m patid->imputed-value frames, fitted kernel or seeds)."""
    t0 = time.time()
    work = pat_frame.drop(columns=["patid"]).copy()
    work = work.apply(pd.to_numeric, errors="coerce").astype("float64")

    target_cols = [c for c in work.columns if work[c].isna().any()]

    # A column with no observed values at all is a merge failure, not missing
    # data. miceforest would otherwise die inside a numpy sampling call with an
    # error that says nothing about which column is at fault.
    _empty = [c for c in target_cols if int(work[c].notna().sum()) == 0]
    if _empty:
        raise ValueError(
            f"These columns are 100% missing, so there is nothing to impute from: "
            f"{_empty}. Fix the source merge before imputing.")
    _sparse = {c: round(100 * work[c].notna().mean(), 2) for c in target_cols
               if work[c].notna().mean() < 0.5}
    if _sparse:
        print(f"[{label}] WARNING: under half observed for {_sparse} (% observed) - "
              "check the merge before trusting these imputations.")

    if verbose:
        print(f"[{label}] imputing {target_cols} | m={m}, iterations={iterations}, "
              f"predictors={work.shape[1] - 1}, n={len(work):,}")
        extra = [c for c in target_cols if c not in continuous_cols]
        if extra:
            print(f"[{label}] note: these predictors also had missing values and "
                  f"were imputed jointly: {extra}")
    if not target_cols:
        print(f"[{label}] nothing to impute - no missing values found.")
        base = pat_frame[["patid"] + [c for c in continuous_cols if c in pat_frame.columns]].copy()
        return [base.copy() for _ in range(m)], None

    if n_jobs and n_jobs > 1 and m > 1:
        return run_mice_parallel(pat_frame, continuous_cols, target_cols, m, iterations,
                                 seed, mean_match_candidates, n_jobs, label=label,
                                 **lgb_kwargs)

    kernel = _make_kernel(work, m, seed, mean_match_candidates)
    kernel.mice(iterations, **lgb_kwargs)

    out = []
    for i in range(m):
        comp = kernel.complete_data(dataset=i)
        frame = pd.DataFrame({"patid": pat_frame["patid"].to_numpy()})
        for c in continuous_cols:
            if c in comp.columns:
                frame[c] = np.asarray(comp[c])
            elif c in pat_frame.columns:
                frame[c] = pat_frame[c].to_numpy()
        out.append(frame)
    print(f"[{label}] MICE finished in {time.time() - t0:.1f}s")
    return out, kernel


def imputation_diagnostics(pat_frame, imputations, continuous_cols):
    """Observed vs imputed distributions - include in the supplement."""
    rows = []
    for c in continuous_cols:
        if c not in pat_frame.columns:
            continue
        miss = pat_frame[c].isna().to_numpy()
        obs = pat_frame.loc[~miss, c]
        rows.append(dict(variable=c, source="observed", n=int(obs.size), mean=obs.mean(),
                         sd=obs.std(), p25=obs.quantile(.25), median=obs.median(),
                         p75=obs.quantile(.75)))
        if miss.sum() == 0:
            continue
        for i, imp in enumerate(imputations, 1):
            v = pd.to_numeric(imp.loc[miss, c])
            rows.append(dict(variable=c, source=f"imputed {i}", n=int(v.size), mean=v.mean(),
                             sd=v.std(), p25=v.quantile(.25), median=v.median(),
                             p75=v.quantile(.75)))
    return pd.DataFrame(rows).round(3)


# ---- 4. Rubin's rules ------------------------------------------------------
def rubin_pool(summaries, n_events=None, n_params=None):
    """Pool lifelines `.summary` frames on the log-hazard scale.

    Returns a frame shaped like a lifelines summary (so it can be passed
    straight to build_bleeding_summary) plus FMI, RIV and Barnard-Rubin df."""
    coef = pd.concat([s["coef"] for s in summaries], axis=1)
    se = pd.concat([s["se(coef)"] for s in summaries], axis=1)
    m = coef.shape[1]
    if m < 2:
        raise ValueError("Rubin pooling needs at least 2 imputed datasets")

    qbar = coef.mean(axis=1)                 # pooled point estimate
    ubar = (se ** 2).mean(axis=1)            # within-imputation variance
    b = coef.var(axis=1, ddof=1)             # between-imputation variance
    total = ubar + (1 + 1 / m) * b           # total variance
    se_pool = np.sqrt(total)

    lam = np.where(total > 0, ((1 + 1 / m) * b) / total, 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(ubar > 0, ((1 + 1 / m) * b) / ubar, 0.0)
        df_old = np.where(r > 0, (m - 1) * (1 + 1 / r) ** 2, np.inf)

    if n_events and n_params:                # Barnard & Rubin (1999) adjustment
        nu_com = max(int(n_events) - int(n_params), 1)
        df_obs = ((nu_com + 1) / (nu_com + 3)) * nu_com * (1 - lam)
        df_adj = np.where(np.isfinite(df_old), 1 / (1 / df_old + 1 / df_obs), float(nu_com))
    else:
        df_adj = np.where(np.isfinite(df_old), df_old, 1e6)
    df_adj = np.clip(df_adj, 1.0, 1e7)

    crit = stats.t.ppf(0.975, df_adj)
    zstat = qbar / se_pool
    out = pd.DataFrame({
        "coef": qbar,
        "se(coef)": se_pool,
        "exp(coef)": np.exp(qbar),
        "exp(coef) lower 95%": np.exp(qbar - crit * se_pool),
        "exp(coef) upper 95%": np.exp(qbar + crit * se_pool),
        "coef lower 95%": qbar - crit * se_pool,
        "coef upper 95%": qbar + crit * se_pool,
        "z": zstat,
        "p": 2 * stats.t.sf(np.abs(zstat), df_adj),
        "fmi": lam + 2 / (df_adj + 3),
        "riv": r,
        "df": df_adj,
        "n_imputations": m,
    })
    out.index.name = "covariate"
    return out


# ---- 5. Drivers: fit across imputations, pool, and complete-case fallback --


# ---------------------------------------------------------------------------
# 6. Multivariate Rubin's rules (pooled parameter vector AND covariance matrix)
# ---------------------------------------------------------------------------
# The scalar pooling above is enough when every reported number is a single
# coefficient. It is NOT enough here: some tables below combine coefficients
# (a main effect plus its interaction term) and need the covariance between
# them. Rubin's rules extend to the whole vector - the total variance matrix is
# T = Ubar + (1 + 1/m) B, where Ubar is the average within-imputation covariance
# matrix and B is the between-imputation covariance of the parameter vectors.
# Pooling only the diagonal would understate the covariance and give wrong
# confidence intervals for any derived contrast.

def rubin_pool_mv(params_list, vcov_list, n_events=None):
    """Pool a parameter vector and its covariance matrix across imputations.

    Returns {"summary", "params", "vcov"} so it is a drop-in replacement for a
    single fit's results."""
    P = pd.concat([pd.Series(p) for p in params_list], axis=1)
    m = P.shape[1]
    if m < 2:
        raise ValueError("Rubin pooling needs at least 2 imputed datasets")
    names = P.index

    qbar = P.mean(axis=1)
    Ubar = sum(v.reindex(index=names, columns=names) for v in vcov_list) / m
    dev = P.sub(qbar, axis=0).to_numpy()
    B = pd.DataFrame(dev @ dev.T / (m - 1), index=names, columns=names)
    T = Ubar + (1 + 1 / m) * B

    u_diag = pd.Series(np.diag(Ubar.to_numpy()), index=names)
    b_diag = pd.Series(np.diag(B.to_numpy()), index=names)
    t_diag = pd.Series(np.diag(T.to_numpy()), index=names)
    se_pool = np.sqrt(t_diag)

    lam = np.where(t_diag > 0, ((1 + 1 / m) * b_diag) / t_diag, 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(u_diag > 0, ((1 + 1 / m) * b_diag) / u_diag, 0.0)
        df_old = np.where(r > 0, (m - 1) * (1 + 1 / r) ** 2, np.inf)
    if n_events:
        nu_com = max(int(n_events) - len(names), 1)
        df_obs = ((nu_com + 1) / (nu_com + 3)) * nu_com * (1 - lam)
        df_adj = np.where(np.isfinite(df_old), 1 / (1 / df_old + 1 / df_obs), float(nu_com))
    else:
        df_adj = np.where(np.isfinite(df_old), df_old, 1e6)
    df_adj = np.clip(df_adj, 1.0, 1e7)

    crit = stats.t.ppf(0.975, df_adj)
    zstat = qbar / se_pool
    summary = pd.DataFrame({
        "coef": qbar,
        "se(coef)": se_pool,
        "exp(coef)": np.exp(qbar),
        "exp(coef) lower 95%": np.exp(qbar - crit * se_pool),
        "exp(coef) upper 95%": np.exp(qbar + crit * se_pool),
        "z": zstat,
        "p": 2 * stats.t.sf(np.abs(zstat), df_adj),
        "fmi": lam + 2 / (df_adj + 3),
        "df": df_adj,
        "n_imputations": m,
    })
    summary.index.name = "covariate"
    return {"summary": summary, "params": qbar, "vcov": T}


In [ ]:
# ---- Adapter: MI driver matching this notebook's fit_models() / fit_strata() -
# fit_models() takes {key: (formula, cols, filter_or_None)}. The MI version runs
# every stratum on every imputed dataset and pools with Rubin's rules. Imputed
# values are assigned BEFORE the stratum filter is applied, so a worker always
# sees a fully populated frame.

IMPUTE_COLS = ["ckd_value", "bmi_value"]
MI_FIT_N_JOBS = FIT_N_JOBS

_MI_BASE_DF = None      # inherited copy-on-write by forked workers
_MI_VALUES = None


def _fit_mi_worker(args):
    imp_idx, key, formula, cols, filt = args
    need = list(cols)
    if filt is not None and filt[0] not in need:
        need = need + [filt[0]]
    sub = _MI_BASE_DF[need].copy()
    for c, arr in _MI_VALUES[imp_idx].items():
        if c in sub.columns:
            sub[c] = arr
    if filt is not None:
        sub = sub[sub[filt[0]] == filt[1]]
    ctv = CoxTimeVaryingFitter()
    ctv.fit(sub[cols], id_col="patid", start_col="start_date", stop_col="end_date",
            event_col="event_flag", formula=formula, show_progress=False)
    return imp_idx, key, ctv.summary, len(sub), int(sub["event_flag"].sum())


def fit_models_mi(df, imputations, specs, impute_cols=None, n_jobs=None, label=""):
    """Same contract as fit_models(), but pooled across imputed datasets."""
    global _MI_BASE_DF, _MI_VALUES
    impute_cols = IMPUTE_COLS if impute_cols is None else list(impute_cols)
    n_jobs = MI_FIT_N_JOBS if n_jobs is None else n_jobs
    t0 = time.time()

    values = []
    for i, imp in enumerate(imputations, 1):
        lut = imp.set_index("patid")
        vals = {}
        for c in impute_cols:
            if c not in lut.columns or c not in df.columns:
                continue
            arr = df["patid"].map(lut[c]).to_numpy(dtype=float)
            n_bad = int(np.isnan(arr).sum())
            if n_bad:
                raise ValueError(f"{c}: {n_bad} rows still NaN after imputation {i} - "
                                 "some patids are absent from the imputation frame.")
            vals[c] = arr
        values.append(vals)

    tasks = [(i, k, f, c, flt) for i in range(len(imputations))
             for k, (f, c, flt) in specs.items()]
    _MI_BASE_DF, _MI_VALUES = df, values
    try:
        with Pool(min(n_jobs, len(tasks))) as pool:
            res = pool.map(_fit_mi_worker, tasks)
    finally:
        _MI_BASE_DF = _MI_VALUES = None

    per_key = {k: {"summaries": [None] * len(imputations), "n_rows": 0, "n_events": 0}
               for k in specs}
    for imp_idx, key, summ, n_rows, n_ev in res:
        per_key[key]["summaries"][imp_idx] = summ
        per_key[key]["n_rows"] = n_rows
        per_key[key]["n_events"] = n_ev

    out = {}
    for key, v in per_key.items():
        pooled = rubin_pool(v["summaries"], n_events=v["n_events"],
                            n_params=len(v["summaries"][0]))
        out[key] = {"summary": pooled, "n_rows": v["n_rows"], "n_events": v["n_events"]}
    print(f"[{label}] {len(tasks)} fits on {min(n_jobs, len(tasks))} workers, "
          f"pooled in {time.time() - t0:.1f}s")
    return out


def fit_strata_mi(df, imputations, group_col, groups, exclude_continuous=(), label=""):
    """MI version of fit_strata(). Returns {group: summary DataFrame}, the same
    shape the downstream table-building cells already expect."""
    f_no, f_yes, comorbs = build_formulas(exclude_continuous)
    cols = (["patid", "start_date", "end_date", "event_flag"]
            + drug_classes_to_consider + comorbs)

    specs = {}
    for g in groups:
        specs[f"{g}|no_ppi"] = (f_no, cols, (group_col, g))
        specs[f"{g}|with_ppi"] = (f_yes, cols + ["PPI_baseline"], (group_col, g))

    # Only impute what the model actually uses: when stratifying on eGFR, the
    # eGFR term is excluded from the formula, so only BMI needs imputing (and
    # vice versa). The stratifier itself always comes from the observed data.
    impute_cols = [c for c in IMPUTE_COLS if c in comorbs]

    fits = fit_models_mi(df, imputations, specs, impute_cols=impute_cols, label=label)
    for g in groups:
        n = fits[f"{g}|no_ppi"]["n_events"]
        print(f"    {group_col}={g}: {fits[f'{g}|no_ppi']['n_rows']:,} rows, {n:,} events")
    return {g: split_summary(fits, g) for g in groups}


def strata_completeness(df, group_col, groups, label=""):
    """How many rows fall outside every named stratum - i.e. are silently dropped
    by the filters. Non-zero is worth understanding before reporting."""
    inside = df[group_col].isin(groups).sum()
    total = len(df)
    print(f"[{label}] {inside:,} of {total:,} rows fall inside {groups}; "
          f"{total - inside:,} dropped")
    return total - inside


## A note on stratifying by an imputed variable

Two of the subgroup analyses below stratify on the very variables that have
missing values: eGFR (`ckd_value`) and BMI (`bmi_value`). That needs a decision,
and the decision is worth stating explicitly in the methods.

**What this notebook does.** The stratifying variable is always taken from the
**observed** data, never from an imputed value. Patients with a missing eGFR are
excluded from the eGFR-stratified analysis, and likewise for BMI - which is what
the original `dropna()` calls already did. Multiple imputation is then used for
the *other* continuous covariate, which still enters the model as an adjustment
term. So the eGFR-stratified models use observed eGFR for the strata and imputed
BMI for adjustment, and vice versa.

**Why.** Imputed values are proper random draws, so letting them decide subgroup
membership would make the subgroups themselves partly synthetic and would change
which patients are in which stratum from one imputed dataset to the next. That
is defensible in principle - Rubin's rules still apply - but it is much harder to
explain to a reviewer, and the subgroup sizes stop being a fixed, reportable
quantity. Keeping the stratifier observed costs some precision within each
stratum and nothing else.

The age analysis is unaffected: age is backfilled from year of birth, not
imputed, so every patient has an age group.


In [ ]:
# ---- MICE for the serious-bleeding cohort ----------------------------------
# One imputation set per outcome. Every subgroup analysis on this cohort reuses
# it; the strata themselves are built from observed values (see the note above).
major_imp_frame = build_imputation_frame(
    major_df, comorbidities,
    binary_comorbidity_cols, continuous_comorbidity_cols,
    drug_terms=drug_classes_to_consider, extra_binary=("PPI_baseline",),
    event_col="event_flag")

print(major_imp_frame[["fu_years", "event", "na_cumhaz"]].describe().round(3))

major_imputations, major_mice_seeds = run_mice(
    major_imp_frame, IMPUTE_COLS, label="Serious bleeding", **MI_CONFIG)

major_imp_diag = imputation_diagnostics(major_imp_frame, major_imputations, IMPUTE_COLS)
major_imp_diag


In [ ]:
### Analysis by age
age_major_df = major_df.copy()
age_major_df["age_group"] = pd.cut(age_major_df["age_at_study_start"], bins=[18, 59, 74, 120], labels=["18-59", "60-74", "75+"], right=True, include_lowest=True)
age_major_df.drop(columns=["age_at_study_start"], inplace=True)

In [ ]:
# Reference only - the fit calls below build their own formulas through
# build_formulas(), so nothing here leaks into a later section.
age_covariates_no_ppi, age_covariates_with_ppi, _ = build_formulas(("age_at_study_start",))
age_covariates_no_ppi


In [ ]:
AGE_GROUPS = ["18-59", "60-74", "75+"]

major_age_summary_list = fit_strata_mi(
    age_major_df, major_imputations, "age_group", AGE_GROUPS,
    exclude_continuous=("age_at_study_start",), label="Age / severe",
)


### Minor Bleeding


In [ ]:
minor_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_minor.xlsx"), dtype=str)
minor_doac_exposure["age_at_study_start"] = minor_doac_exposure["age_at_study_start"].astype(int)
minor_doac_exposure["study_start_date"] = pd.to_datetime(minor_doac_exposure["study_start_date"])
minor_doac_exposure["study_end_date"] = pd.to_datetime(minor_doac_exposure["study_end_date"])
minor_doac_exposure["event_date"] = pd.to_datetime(minor_doac_exposure["event_date"])


#doac_exposure["event_date"] = pd.to_datetime(doac_exposure["event_date"])

minor_doac_exposure = minor_doac_exposure[(minor_doac_exposure["age_at_study_start"] >= 18) &
                (minor_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (minor_doac_exposure["warfarin_1w_before"] =="0") &
                    (minor_doac_exposure["warfarin_during_study"] =="0")]

# age_at_study_start is retained here: it is the authoritative source for the
# backfill in the covariate cell below. Dropping it forced age to be re-derived
# from the comorbidity workbook, which is where the NaNs came from.
minor_doac_exposure = minor_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date", "age_at_study_start"]]

In [ ]:
minor_interacting_drugs_presc = interacting_drugs_exposure.copy()
minor_interacting_drugs_presc["study_end_date"] = minor_interacting_drugs_presc["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])
minor_interacting_drugs_presc = minor_interacting_drugs_presc[minor_interacting_drugs_presc["issuedate"] < minor_interacting_drugs_presc["study_end_date"]]
minor_interacting_drugs_presc.sort_values(['patid', 'issuedate'], inplace=True)
minor_interacting_drugs_presc.reset_index(drop=True, inplace=True)
minor_interacting_drugs_presc

In [ ]:
minor_interacting_drugs_presc_merged = merge_prescriptions(minor_interacting_drugs_presc)
minor_interacting_drugs_presc_merged["study_start_date"] = minor_interacting_drugs_presc_merged["patid"].map(minor_doac_exposure.set_index("patid")["study_start_date"])
minor_interacting_drugs_presc_merged["study_end_date"] = minor_interacting_drugs_presc_merged["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])
#interacting_drugs_presc_merged["event_date"] = interacting_drugs_presc_merged["patid"].map(events_df.set_index("patid")["event_date"])
minor_interacting_drugs_presc_merged

In [ ]:
minor_events_df = minor_doac_exposure[minor_doac_exposure["event_date"].notnull()]

minor_exposure_timeline_df = build_exposure_timeline(minor_interacting_drugs_presc_merged)
minor_exposure_timeline_df["study_start_date"] = minor_exposure_timeline_df["patid"].map(minor_doac_exposure.set_index("patid")["study_start_date"])
minor_exposure_timeline_df["study_end_date"] = minor_exposure_timeline_df["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])

minor_exposure_timeline_df = minor_exposure_timeline_df.merge(minor_events_df[["patid", "event_date"]], on="patid", how="left")
minor_exposure_timeline_df["event_flag"] = np.where(
    (minor_exposure_timeline_df["event_date"] >= minor_exposure_timeline_df["start_date"]) & 
    (minor_exposure_timeline_df["event_date"] <= minor_exposure_timeline_df["end_date"]), 1, 0)

In [ ]:
pats_with_exposures = set(minor_exposure_timeline_df["patid"].unique())

unique_minor_events_df = minor_events_df.drop_duplicates(subset=["patid"]).copy()

pats_without_exposures_df = minor_doac_exposure[~minor_doac_exposure["patid"].isin(pats_with_exposures)].copy()
pats_without_exposures_df["concomitant_drugs_ls"] = [[] for _ in range(len(pats_without_exposures_df))]
pats_without_exposures_df["start_date"] = pats_without_exposures_df["study_start_date"]
pats_without_exposures_df["end_date"] = pats_without_exposures_df["study_end_date"]
pats_without_exposures_df["event_date"] = pats_without_exposures_df["patid"].map(
    unique_minor_events_df.set_index("patid")["event_date"]
)
pats_without_exposures_df["event_flag"] = np.where(
    (pats_without_exposures_df["event_date"] >= pats_without_exposures_df["start_date"]) &
    (pats_without_exposures_df["event_date"] <= pats_without_exposures_df["end_date"]),
    1, 0
)

minor_exposure_timeline_df = pd.concat([minor_exposure_timeline_df, pats_without_exposures_df], ignore_index=True)
minor_exposure_timeline_df.reset_index(drop=True, inplace=True)

In [ ]:
minor_df = minor_exposure_timeline_df.copy()

for drug in drug_classes_to_consider: #+ ["PPI"]:
    minor_df[f"{drug}"] = minor_df["concomitant_drugs_ls"].apply(lambda drugs: 1 if drug in drugs else 0)

for col in binary_comorbidity_cols:
    minor_df[col] = minor_df["patid"].map(comorbidities.set_index("patid")[col]).fillna(0).astype(int)

for col in continuous_comorbidity_cols:
    minor_df[col] = minor_df["patid"].map(comorbidities.set_index("patid")[col])

minor_df = minor_df[minor_df["start_date"] < minor_df["end_date"]].reset_index(drop=True)

minor_df["PPI_baseline"] = minor_df["patid"].isin(ppi_pats).astype(int)

In [ ]:
# ---- MICE for the minor-bleeding cohort ------------------------------------
minor_df = backfill_age(minor_df, minor_doac_exposure, "Minor bleeding")
missing_report_minor = report_missingness(minor_df, "Minor bleeding")

minor_imp_frame = build_imputation_frame(
    minor_df, comorbidities,
    binary_comorbidity_cols, continuous_comorbidity_cols,
    drug_terms=drug_classes_to_consider, extra_binary=("PPI_baseline",),
    event_col="event_flag")

minor_imputations, minor_mice_seeds = run_mice(
    minor_imp_frame, IMPUTE_COLS, label="Minor bleeding", **MI_CONFIG)

minor_imp_diag = imputation_diagnostics(minor_imp_frame, minor_imputations, IMPUTE_COLS)
minor_imp_diag


In [ ]:
age_minor_df = minor_df.copy()
# age_at_study_start is already backfilled on minor_df, so it no longer needs to
# be re-read from the comorbidity workbook (that re-read was what reintroduced
# the NaNs, and the .astype(int) below then failed on them).
age_minor_df["age_group"] = pd.cut(age_minor_df["age_at_study_start"], bins=[18, 59, 74, 120],
                                   labels=["18-59", "60-74", "75+"], right=True, include_lowest=True)
strata_completeness(age_minor_df, "age_group", AGE_GROUPS, label="Age / minor")


In [ ]:
minor_age_summary_list = fit_strata_mi(
    age_minor_df, minor_imputations, "age_group", AGE_GROUPS,
    exclude_continuous=("age_at_study_start",), label="Age / minor",
)


In [ ]:
rows = []

for age_group in ["18-59", "60-74", "75+"]:

    for type_name, summary_dict in [
        ("Severe", major_age_summary_list),
        ("Minor", minor_age_summary_list)
    ]:

        summary = summary_dict[age_group]

        row = {
            "age_group": age_group,
            "type": type_name
        }

        for _, r in summary.iterrows():
            row[r["covariate"]] = (
                f"{r['HR']:.2f} "
                f"({r['CI Lower']:.2f}-{r['CI Upper']:.2f})"
            )

        rows.append(row)

age_hr_table = pd.DataFrame(rows)

# Optional: order columns
age_hr_table = age_hr_table[
    ["age_group", "type"] + drug_classes_to_consider
]

age_hr_table

In [ ]:
def plot_forest_from_table(
    df: pd.DataFrame,
    group_cols: list,
    value_cols: list,
    figsize_per_plot: float = 3,
    height: float = 6
):
    forest_df = df.melt(
        id_vars=group_cols,
        value_vars=value_cols,
        var_name="Variable",
        value_name="HR_CI"
    )

    # Parse HR + CI
    forest_df[["HR", "CI_lower", "CI_upper"]] = (
        forest_df["HR_CI"]
        .str.extract(r'([\d.]+)\s*\(([\d.]+)-([\d.]+)\)')
        .astype(float)
    )

    forest_df["Group"] = forest_df[group_cols].astype(str).agg(" | ".join, axis=1)
    forest_df = forest_df.sort_values(group_cols)

    group_order = forest_df["Group"].drop_duplicates().tolist()

    forest_df["Group"] = pd.Categorical(
        forest_df["Group"],
        categories=group_order,
        ordered=True
    )
    n_vars = len(value_cols)

    fig, axes = plt.subplots(
        1,
        n_vars,
        figsize=(figsize_per_plot * n_vars, height),
        sharey=True
    )

    if n_vars == 1:
        axes = [axes]

    # -----------------------------
    # PLOT EACH VARIABLE
    # -----------------------------
    for ax, var in zip(axes, value_cols):

        df_plot = (
            forest_df[forest_df["Variable"] == var]
            .sort_values("Group")
            .reset_index(drop=True)
        )

        y = np.arange(len(df_plot))[::-1]

        ax.errorbar(
            x=df_plot["HR"],
            y=y,
            xerr=[
                df_plot["HR"] - df_plot["CI_lower"],
                df_plot["CI_upper"] - df_plot["HR"]
            ],
            fmt='o',
            capsize=3
        )

        ax.axvline(1, linestyle="--", linewidth=1)
        ax.set_title(var)
        ax.set_xlabel("HR")

        ax.set_yticks(y)
        ax.set_yticklabels(df_plot["Group"])

    plt.tight_layout()
    plt.show()

    return fig, axes

In [ ]:
plot_forest_from_table(
    age_hr_table,
    group_cols=["age_group", "type"],
    value_cols=[
        "NSAID", "Antiplatelet", "SSRI",
        "CCB", "SNRI", "Antiarrythmic"
    ]
)

## Analysis by CKG

In [ ]:
ckd_major_df = major_df.copy()
ckd_major_df["ckd_stage"] = pd.cut(
    ckd_major_df["ckd_value"],
    bins=[0, 45, 90, np.inf],
    labels=[ "<45", "45-89", "90+"],
    right=False
)
# The stratifier stays OBSERVED - imputed eGFR is never used to decide which
# stratum a patient belongs to. eGFR is excluded from the model formula within
# strata, so this dropna costs nothing beyond the stratum definition itself.
_before = ckd_major_df["patid"].nunique()
ckd_major_df.dropna(subset=["ckd_value"], inplace=True)
print(f"CKD strata (severe): {ckd_major_df['patid'].nunique():,} of {_before:,} patients "
      f"have an observed eGFR")


In [ ]:
ckd_major_df["ckd_stage"].value_counts()

In [ ]:
ckd_minor_df = minor_df.copy()
ckd_minor_df["ckd_stage"] = pd.cut(
    ckd_minor_df["ckd_value"],
    bins=[0, 45, 90, np.inf],
    labels=[ "<45", "45-89", "90+"],
    right=False
)
# The dropna was commented out here but active for the severe cohort. Rows with a
# missing eGFR got a NaN stratum and were dropped by the filter anyway, so this
# only makes the two cohorts consistent and makes the count reportable.
_before = ckd_minor_df["patid"].nunique()
ckd_minor_df.dropna(subset=["ckd_value"], inplace=True)
print(f"CKD strata (minor): {ckd_minor_df['patid'].nunique():,} of {_before:,} patients "
      f"have an observed eGFR")


In [ ]:
ckd_covariates_no_ppi, ckd_covariates_with_ppi, _ = build_formulas(("ckd_value",))
ckd_covariates_no_ppi


In [ ]:
CKD_GROUPS = ["<45", "45-89", "90+"]

major_ckd_summary_list = fit_strata_mi(
    ckd_major_df, major_imputations, "ckd_stage", CKD_GROUPS,
    exclude_continuous=("ckd_value",), label="CKD / severe",
)


In [ ]:
minor_ckd_summary_list = fit_strata_mi(
    ckd_minor_df, minor_imputations, "ckd_stage", CKD_GROUPS,
    exclude_continuous=("ckd_value",), label="CKD / minor",
)


In [ ]:
rows = []

for ckd_group in ["<45", "45-89", "90+"]:

    for type_name, summary_dict in [
        ("Severe", major_ckd_summary_list),
        ("Minor", minor_ckd_summary_list)
    ]:

        summary = summary_dict[ckd_group]

        row = {
            "ckd_group": ckd_group,
            "type": type_name
        }

        for _, r in summary.iterrows():
            row[r["covariate"]] = (
                f"{r['HR']:.2f} "
                f"({r['CI Lower']:.2f}-{r['CI Upper']:.2f})"
            )

        rows.append(row)

ckd_hr_table = pd.DataFrame(rows)

# Optional: order columns
ckd_hr_table = ckd_hr_table[
    ["ckd_group", "type"] + drug_classes_to_consider
]

ckd_hr_table

In [ ]:
plot_forest_from_table(
    ckd_hr_table,
    group_cols=["ckd_group", "type"],
    value_cols=[
        "NSAID", "Antiplatelet", "SSRI",
        "CCB", "SNRI", "Antiarrythmic"
    ]
)

### Analysis by BMI

In [ ]:
bmi_major_df = major_df.copy()
bmi_major_df["bmi_category"] = pd.cut(
    bmi_major_df["bmi_value"],
    bins=[0, 18.5, 25, 30, 35],
    labels=["Underweight", "Normal weight", "Overweight", "Obese"],
    right=False
)
# WARNING: the top bin closes at 35, so anyone with BMI >= 35 gets a NaN category
# and is silently excluded from every BMI stratum - that is obesity class II and
# III, not a rare group. The eGFR bins above use np.inf for exactly this reason.
# Left as-is because it changes the analysis; the count is printed so the
# decision is at least visible. Use bins=[0, 18.5, 25, 30, np.inf] to include them.
_n_high = int((pd.to_numeric(bmi_major_df["bmi_value"], errors="coerce") >= 35).sum())
print(f"BMI strata (severe): {_n_high:,} rows have BMI >= 35 and fall outside every stratum")

# The stratifier stays OBSERVED - imputed BMI is never used to assign a category.
_before = bmi_major_df["patid"].nunique()
bmi_major_df.dropna(subset=["bmi_value"], inplace=True)
print(f"BMI strata (severe): {bmi_major_df['patid'].nunique():,} of {_before:,} patients "
      f"have an observed BMI")


In [ ]:
bmi_minor_df = minor_df.copy()
bmi_minor_df["bmi_category"] = pd.cut(
    bmi_minor_df["bmi_value"],
    bins=[0, 18.5, 25, 30, 35],
    labels=["Underweight", "Normal weight", "Overweight", "Obese"],
    right=False
)
# WARNING: the top bin closes at 35, so anyone with BMI >= 35 gets a NaN category
# and is silently excluded from every BMI stratum - that is obesity class II and
# III, not a rare group. The eGFR bins above use np.inf for exactly this reason.
# Left as-is because it changes the analysis; the count is printed so the
# decision is at least visible. Use bins=[0, 18.5, 25, 30, np.inf] to include them.
_n_high = int((pd.to_numeric(bmi_minor_df["bmi_value"], errors="coerce") >= 35).sum())
print(f"BMI strata (minor): {_n_high:,} rows have BMI >= 35 and fall outside every stratum")

# The stratifier stays OBSERVED - imputed BMI is never used to assign a category.
_before = bmi_minor_df["patid"].nunique()
bmi_minor_df.dropna(subset=["bmi_value"], inplace=True)
print(f"BMI strata (minor): {bmi_minor_df['patid'].nunique():,} of {_before:,} patients "
      f"have an observed BMI")


In [ ]:
bmi_covariates_no_ppi, bmi_covariates_with_ppi, _ = build_formulas(("bmi_value",))
bmi_covariates_no_ppi


In [ ]:
BMI_GROUPS = ["Underweight", "Normal weight", "Overweight", "Obese"]

major_bmi_summary_list = fit_strata_mi(
    bmi_major_df, major_imputations, "bmi_category", BMI_GROUPS,
    exclude_continuous=("bmi_value",), label="BMI / severe",
)


In [ ]:
minor_bmi_summary_list = fit_strata_mi(
    bmi_minor_df, minor_imputations, "bmi_category", BMI_GROUPS,
    exclude_continuous=("bmi_value",), label="BMI / minor",
)


In [ ]:
rows = []

for bmi_group in ["Underweight", "Normal weight", "Overweight", "Obese"]:

    for type_name, summary_dict in [
        ("Severe", major_bmi_summary_list),
        ("Minor", minor_bmi_summary_list)
    ]:

        summary = summary_dict[bmi_group]

        row = {
            "bmi_group": bmi_group,
            "type": type_name
        }

        for _, r in summary.iterrows():
            row[r["covariate"]] = (
                f"{r['HR']:.2f} "
                f"({r['CI Lower']:.2f}-{r['CI Upper']:.2f})"
            )

        rows.append(row)

bmi_hr_table = pd.DataFrame(rows)

# Optional: order columns
bmi_hr_table = bmi_hr_table[
    ["bmi_group", "type"] + drug_classes_to_consider
]

bmi_hr_table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_forest_from_table(
    df: pd.DataFrame,
    group_cols: list,
    value_cols: list,
    figsize_per_plot: float = 3,
    height: float = 6
):
    """
    Fully dynamic forest plot generator.

    Parameters
    ----------
    df : DataFrame
        Wide table with group columns + value columns formatted as
        'HR (CI_low-CI_high)'.

    group_cols : list
        Columns used for grouping (e.g. ['age_group', 'type']).
        Order matters: first = main group, second = subgroup.

    value_cols : list
        Columns to plot (drug classes etc).

    """

    # -----------------------------
    # LONG FORMAT
    # -----------------------------
    forest_df = df.melt(
        id_vars=group_cols,
        value_vars=value_cols,
        var_name="Variable",
        value_name="HR_CI"
    )

    # Parse HR + CI
    forest_df[["HR", "CI_lower", "CI_upper"]] = (
        forest_df["HR_CI"]
        .str.extract(r'([\d.]+)\s*\(([\d.]+)-([\d.]+)\)')
        .astype(float)
    )

    # -----------------------------
    # DYNAMIC GROUP LABEL
    # -----------------------------
    forest_df["Group"] = forest_df[group_cols].astype(str).agg(" | ".join, axis=1)

    # enforce ordering: stable appearance order
    forest_df = forest_df.sort_values(group_cols)

    group_order = forest_df["Group"].drop_duplicates().tolist()

    forest_df["Group"] = pd.Categorical(
        forest_df["Group"],
        categories=group_order,
        ordered=True
    )

    # -----------------------------
    # PLOT SETUP
    # -----------------------------
    n_vars = len(value_cols)

    fig, axes = plt.subplots(
        1,
        n_vars,
        figsize=(figsize_per_plot * n_vars, height),
        sharey=True
    )

    if n_vars == 1:
        axes = [axes]

    # -----------------------------
    # PLOT EACH VARIABLE
    # -----------------------------
    for ax, var in zip(axes, value_cols):

        df_plot = (
            forest_df[forest_df["Variable"] == var]
            .sort_values("Group")
            .reset_index(drop=True)
        )

        y = np.arange(len(df_plot))[::-1]

        ax.errorbar(
            x=df_plot["HR"],
            y=y,
            xerr=[
                df_plot["HR"] - df_plot["CI_lower"],
                df_plot["CI_upper"] - df_plot["HR"]
            ],
            fmt='o',
            capsize=3
        )

        ax.axvline(1, linestyle="--", linewidth=1)
        ax.set_title(var)
        ax.set_xlabel("HR")

        ax.set_yticks(y)
        ax.set_yticklabels(df_plot["Group"])

    plt.tight_layout()
    plt.show()

    return fig, axes

In [ ]:
plot_forest_from_table(
    bmi_hr_table,
    group_cols=["bmi_group", "type"],
    value_cols=[
        "NSAID", "Antiplatelet", "SSRI",
        "CCB", "SNRI", "Antiarrythmic"
    ]
)